# Marker analysis

This notebook computes configured RNA and protein marker availability, marker ROC-AUC,
mean target-cell marker expression by fraction, and fraction-level marker summaries.

Marker names come from `config/marker_config.yaml`.  Markers not present in the dataset
are flagged as unavailable; their absence does not affect the benchmark metrics.

Mirrors: `scripts/run_marker_analysis.py`

In [ ]:
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
for _p in [PROJECT_ROOT, PROJECT_ROOT / "src"]:
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
assert (PROJECT_ROOT / "src" / "rarecell").exists(), PROJECT_ROOT
PROJECT_ROOT

## Required inputs

| Input | Description |
|---|---|
| `config/benchmark_config.yaml` | Benchmark config: `dataset_path`, `label_column`, `fractions`, `seeds`, `representations` |
| `config/marker_config.yaml` | Marker gene/protein lists per cell type (RNA and protein, keyed by cell-type name) |
| `data/processed/pbmc5k_10x_citeseq_representations.h5ad` | Benchmark-ready AnnData with `obsm` PCA embeddings and raw `uns['protein_names']` (produced by `baseline_representations.ipynb`) |

The `--skip-by-fraction` flag can be passed to `run_marker_analysis.py` to skip the
computationally expensive fraction-level AUC computation.

## Canonical script command

```bash
python scripts/run_marker_analysis.py \
    --config config/benchmark_config.yaml \
    --marker-config config/marker_config.yaml
```

Key functions used internally:
- `rarecell.markers.load_marker_config(path)` — loads and validates `marker_config.yaml`
- `rarecell.markers.compute_marker_summary_table(adata, label_column, targets, marker_config)` — per-marker presence, AUC, target mean, rest mean
- `rarecell.markers.compute_marker_auc_by_fraction(adata, ...)` — AUC recomputed at each downsampling fraction/seed
- `rarecell.markers.write_marker_outputs(summary, by_fraction, output_dir)` — writes all CSVs and PNG figures

In [ ]:
result = subprocess.run(
    [
        sys.executable,
        "scripts/run_marker_analysis.py",
        "--config", "config/benchmark_config.yaml",
        "--marker-config", "config/marker_config.yaml",
    ],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(result.returncode)

## Expected outputs

| File | Description |
|---|---|
| `results/tables/marker_gene_summary.csv` | Per-target, per-RNA-marker: present flag, AUC, mean expression in target vs. rest |
| `results/tables/marker_protein_summary.csv` | Same schema for protein (ADT) markers |
| `results/tables/marker_auc_by_fraction.csv` | AUC recomputed at each (target, representation, fraction, seed, marker) condition |
| `results/figures/marker_auc_by_fraction.png` | Mean marker AUC vs. retained fraction, one line per marker |
| `results/figures/target_marker_expression_by_fraction.png` | Mean target-cell marker expression vs. retained fraction |

In [ ]:
outputs = [
    "results/tables/marker_gene_summary.csv",
    "results/tables/marker_protein_summary.csv",
    "results/tables/marker_auc_by_fraction.csv",
    "results/figures/marker_auc_by_fraction.png",
    "results/figures/target_marker_expression_by_fraction.png",
]
[(path, (PROJECT_ROOT / path).exists()) for path in outputs]

## Summary

In [ ]:
print("Generated outputs:")
for path in outputs:
    p = PROJECT_ROOT / path
    status = "OK" if p.exists() else "MISSING"
    print(f"  [{status}] {path}")
print()
print("Deviations from run_marker_analysis.py:")
print("  - No file-level logging (script uses Python logging to stdout).")
print("  - Data file must exist at dataset_path in config; no interactive fallback.")